# 02. Contextual Feature Engineering

**Goal:** Transform lists of tokens into lists of feature dictionaries. In classical NER, we look at the word itself, its suffixes, whether it's capitalized, and vitally: **what words surround it**.

In [1]:
import json
import joblib

def load_jsonl(filepath):
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

train_data = load_jsonl('../data/raw/train.jsonl')
test_data = load_jsonl('../data/raw/test.jsonl')

> **📌 Decision Note — Why Feature Engineering Method?**
>
> **Chosen approach:** Context Window Dictionaries
>
> **Why this works:** We extract explicitly defined features for every token (is_upper, suffix, previous_word, next_word).
>
> **Alternatives we could have used:**
> | Option | Pros | Cons |
> |--------|------|------|
> | Word Embeddings (Word2Vec) | Captures semantics | Requires averaging/summing which loses token-level precision unless used in a BiLSTM. |
> | Raw Bag-of-Words | Simple | Destroys sequence order entirely. In NER, order is everything (e.g., 'New York'). |
>
> **Why we chose this over alternatives:** For CRFs, extracting localized contextual windows as discrete boolean/string features is the most robust classical approach.

## 1. Feature Extraction Function
This is the secret sauce of Classical NER! We look at word `i` and extract features not just about `i`, but about `i-1` and `i+1`.

In [2]:
def word2features(sent_tokens, sent_pos, i):
    word = sent_tokens[i]
    postag = sent_pos[i]

    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'postag': postag,
        'postag[:2]': postag[:2],
    }
    if i > 0:
        word1 = sent_tokens[i-1]
        postag1 = sent_pos[i-1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
            '-1:postag': postag1,
            '-1:postag[:2]': postag1[:2],
        })
    else:
        features['BOS'] = True # Beginning of sentence

    if i < len(sent_tokens)-1:
        word1 = sent_tokens[i+1]
        postag1 = sent_pos[i+1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
            '+1:postag': postag1,
            '+1:postag[:2]': postag1[:2],
        })
    else:
        features['EOS'] = True # End of sentence

    return features

## 2. Apply to Dataset

In [3]:
def extract_features(data):
    X = []
    y = []
    for sentence in data:
        tokens = sentence['tokens']
        pos = sentence['pos_tags']
        ner = sentence['ner_tags']
        
        # Convert sentence into list of dicts
        sent_features = [word2features(tokens, pos, i) for i in range(len(tokens))]
        X.append(sent_features)
        y.append(ner)
    return X, y

X_train, y_train = extract_features(train_data)
X_test, y_test = extract_features(test_data)

print(f'Sentence 1, Token 1 Features:\n{X_train[0][0]}')

Sentence 1, Token 1 Features:
{'bias': 1.0, 'word.lower()': 'eu', 'word[-3:]': 'EU', 'word[-2:]': 'EU', 'word.isupper()': True, 'word.istitle()': False, 'word.isdigit()': False, 'postag': 'NNP', 'postag[:2]': 'NN', 'BOS': True, '+1:word.lower()': 'rejects', '+1:word.istitle()': False, '+1:word.isupper()': False, '+1:postag': 'VBZ', '+1:postag[:2]': 'VB'}


In [4]:
joblib.dump((X_train, y_train, X_test, y_test), '../data/processed/crf_data.pkl')
print('Saved extracted features.')

Saved extracted features.
